# SAFS check — rebuild the San Andreas deck from raw data, end to end

This is the **real-data** counterpart to `deck_workflow.ipynb`. It runs the whole chain on
the San Andreas fault system using the actual community models, and compares what it builds
against a deck we already run on Frontera.

Everything lands in its own folder: **`outputs/safs_check/`**. Nothing here touches the
demo, and nothing overwrites a shipped deck.

### Before you start

```bash
python tools/stage_safs_data.py          # ~129 MB, gitignored
```

That collects the raw products from the legacy tree into `data/safs_alt/`:

| staged as | what it is | size |
|:--|:--|:--|
| `raw/CSM_orientation.csv` | Yang & Hauksson YHSM-2013 community stress model | 8.6 MB |
| `raw/cvm/CVM_*_h_data.csv` | 37 slices of the statewide "muscal" CVM (Vp, Vs, ρ), 0–70 km | 25 MB |
| `raw/ctm/CTM_*_h_data_final.csv` | 105 slices of the SCEC Community Thermal Model, 0–21 km | 36 MB |
| `mesh_alt.puml.h5` | the ALT fault mesh | 59 MB |

**Runtime: roughly 4–6 minutes**, dominated by the two Delaunay interpolations (the CVM is
37 slices onto a 452 × 361 grid; the CTM is 105 slices). This is not the 5-minute demo —
it is the real thing.

## [0] Setup

In [ ]:
# ---- PARAMETERS ----------------------------------------------------------------
PROJECT      = "safs_alt"
RAW_DATA_DIR = None     # None -> data/safs_alt/ (what stage_safs_data.py fills)
OUT_NAME     = "safs_check"
# Compare against a shipped deck?  Point this at one, or None to skip the comparison.
SHIPPED_DECK = ("~/Downloads/seisol_quakeworx/"
                "safs_seisol_v4_0_0_RSSRW_ALT_THERMAL_CASE1_intermediate"
                "_plast_phi30_40_gradedfw_k1p70_nwredM7p8_sefw0_attenuation_deep40km")
# ----------------------------------------------------------------------------------
import time
from pathlib import Path
import numpy as np
from deckbuild.bootstrap import init
from deckbuild.config import Project

B = init(project=PROJECT, require_files=False)
kw = {} if RAW_DATA_DIR is None else {"data_dir": Path(RAW_DATA_DIR).expanduser()}
cfg = Project.load(B.root / "projects" / f"{PROJECT}.yaml", require_files=True, **kw)
OUT = B.root / "outputs" / OUT_NAME
DECK = B.root / "decks" / f"{PROJECT}_check"
for sub in ("material", "stress", "friction"):
    (OUT / sub).mkdir(parents=True, exist_ok=True)
shipped = Path(SHIPPED_DECK).expanduser() if SHIPPED_DECK else None
if shipped and not shipped.is_dir():
    print(f"  ! shipped deck not found at {shipped}; comparisons will be skipped")
    shipped = None
print(cfg.summary()); print(f"outputs      : {OUT}")

## [1] What the shipped deck used

Before rebuilding anything, read the design straight out of the deck we are targeting. The
introspector recovers it from the files themselves — including three values that exist
**only** as literals inside emitted Lua.

In [ ]:
from deckbuild.introspect import introspect_deck
facts = introspect_deck(shipped) if shipped else {}
if facts:
    for k in ("k_from_filename", "freeze_from_filename", "fw_values",
              "fw_boundaries_s_km", "hypocenter_xyz", "nucleation_radius_m",
              "nucleation_amplitude", "phi_deg_from_field", "rs_b_constant",
              "strike_azimuth_deg", "strike_origin_xy", "Plasticity", "FreqCentral"):
        if k in facts:
            print(f"  {k:22} = {facts[k]}")
    print(f"\n  {len(facts)} fields recovered, {len(facts.unknown)} unknown")

## [2] Mesh — ingest and gate

The ALT fault mesh, at production scale. This ingests and gates it; it does not build it —
see `MESHING.md`.

In [ ]:
from deckbuild.mesh import MeshStage
from deckbuild.geometry import load_fault, snap_hypocenter
t0 = time.time()
mesh_art = MeshStage().build(cfg, OUT)
fault = load_fault(cfg.mesh(), cfg.data_dir, strike=cfg.strike)
snap = snap_hypocenter(cfg.hypocenter(), fault, cfg.strike, cfg.crs,
                       gate_bands=cfg.gate_bands)
print(f"{len(fault):,} fault facets in {time.time()-t0:.0f}s")
snap.report.print()

## [3] Material — 37 raw CVM slices → nc, + Sv(z)

The two-stage recipe: reproject each slice, interpolate onto the inscribed UTM grid, form
the moduli **at the source nodes**, then resample in z.

Plasticity is opt-in here too — the shipped deck used φ = 30/40, which is **not** the
Roten paper's 35/45.

In [ ]:
# ---- PARAMETERS ----------------------------------------------------------------
WITH_PLASTICITY  = True     # the shipped deck has Plasticity = 1
PHI_SOFT, PHI_HARD = 30.0, 40.0   # the SAFS production values, NOT Roten's 35/45
WITH_ATTENUATION = True     # the shipped deck is a viscoelastic run
# ----------------------------------------------------------------------------------
from deckbuild.material import AttenuationSpec, MaterialStage, PlasticitySpec
t0 = time.time()
mstage = MaterialStage()
mat = mstage.build(cfg, OUT / "material",
                   plasticity=PlasticitySpec(PHI_SOFT, PHI_HARD) if WITH_PLASTICITY else None,
                   attenuation=AttenuationSpec() if WITH_ATTENUATION else None)
print(f"material + thermal built in {time.time()-t0:.0f}s")
mstage.verify(cfg, mat, plasticity_artifact=mat.plasticity).print()

In [ ]:
# Compare the rebuilt CVM against the shipped one.
from deckbuild.asagi import asagi_axes, read_asagi
if shipped:
    s_nc = next(shipped.glob("*material_cvm.nc"), None)
    if s_nc:
        gx, gy, gz = asagi_axes(mat.material.path)
        sx, sy, sz = asagi_axes(s_nc)
        print(f"  shape   built {len(gx)}x{len(gy)}x{len(gz)}   "
              f"shipped {len(sx)}x{len(sy)}x{len(sz)}")
        for n, a, b in (("x", gx, sx), ("y", gy, sy), ("z", gz, sz)):
            print(f"  {n} axis identical: {np.array_equal(a, b)}")
        if (len(gx), len(gy), len(gz)) == (len(sx), len(sy), len(sz)):
            _, _, _, bf, _ = read_asagi(mat.material.path, fields=["mu", "rho"])
            _, _, _, sf, _ = read_asagi(s_nc, fields=["mu", "rho"])
            d = np.abs(bf["mu"] - sf["mu"])
            rel = d / np.maximum(np.abs(sf["mu"]), 1e-30)
            print(f"  mu   max|d| {d.max():.4g} Pa  max rel {rel.max():.3e}  "
                  f"median rel {np.median(rel):.3e}")
            print(f"  rho  exactly equal on {np.mean(bf['rho']==sf['rho'])*100:.2f}% of nodes")
            print("\n  NOTE: the grid reproduces exactly; the VALUES do not yet.  See")
            print("  docs/EXERCISE_safs_reproduction.md -- this is an OPEN question, and")
            print("  the workflow does not claim reproduction until it is closed.")

## [4] Stress — CSM orientation × Sv × k = 1.70

The shipped deck uses a **constant** k = 1.70 and, per its filename (verified against the
field below), **no** shallow freeze.

In [ ]:
# ---- PARAMETERS ----------------------------------------------------------------
K_VALUES       = [1.70]     # the shipped deck's closure ratio
K_BOUNDARIES   = []
FREEZE_ABOVE_M = 0.0        # the shipped nc has NO _freeze tag -> the freeze is OFF
# ----------------------------------------------------------------------------------
from deckbuild.stress import KDesign, StressStage
t0 = time.time()
sstage = StressStage()
kdes = KDesign(k_values=tuple(K_VALUES),
               boundaries_s_km=tuple(tuple(b) for b in K_BOUNDARIES))
stress = sstage.build(cfg, OUT / "stress", sv_profile=mat.sv_profile, design=kdes,
                      freeze_above_depth_m=FREEZE_ABOVE_M)
print(f"stress built in {time.time()-t0:.0f}s")
sstage.verify(cfg, stress, design=kdes).print()

## [5] Friction — CTM temperature, CASE 1, and the 7-region graded f_w

The shipped deck is CASE 1 (a velocity-strengthening shallow lid) with a seven-region
`f_w` design — recovered in section [1] straight out of its Lua.

In [ ]:
# ---- PARAMETERS ----------------------------------------------------------------
CASE = 1     # CASE 1 = VS shallow / VW seismogenic / VS deep (see deck_workflow.ipynb)
# Take the f_w design from the shipped deck when we have it, else the 7-region default.
FW_VALUES     = facts.get("fw_values", [0.0, 0.0, 0.045, 0.03, 0.06, 0.0175, 0.05])
FW_BOUNDARIES = facts.get("fw_boundaries_s_km",
                          [(20, 28), (150, 160), (176, 182), (188, 194),
                           (214, 226), (244, 252)])
F0 = 0.6     # the rate-and-state reference friction; f_w must satisfy 0 <= f_w < f0
# ----------------------------------------------------------------------------------
from deckbuild.friction import FrictionStage, FwDesign, nucleation_lua
t0 = time.time()
fstage = FrictionStage()
friction = fstage.build(cfg, OUT / "friction", case=CASE, thermal=mat.thermal)
fstage.verify(cfg, friction).print()

fwdes = FwDesign(fw_values=tuple(float(v) for v in FW_VALUES),
                 boundaries_s_km=tuple(tuple(map(float, b)) for b in FW_BOUNDARIES),
                 f0=F0)
fw = fstage.build_fw_map(cfg, OUT / "friction", fwdes)
fstage.verify_fw_map(cfg, fw, fwdes).print()
print(f"friction built in {time.time()-t0:.0f}s")

In [ ]:
# Does our emitted rs_muw Lua match the shipped one?
# The shipped maps use the LEGACY form `X + inc * g`; ours writes `X + inc`.  Both are
# read by _eval_lua_text, so this compares the FUNCTIONS, not the text.
if shipped:
    s_lua = next(shipped.glob("*rs_muw*.yaml"), None)
    if s_lua:
        from deckbuild.friction import _eval_lua_text
        ss = np.linspace(-50, 500, 1101)
        try:
            a = _eval_lua_text(Path(fw.path).read_text(), ss)
            b = _eval_lua_text(s_lua.read_text(), ss)
            print(f"  shipped: {s_lua.name}")
            print(f"  f_w(s) max |ours - shipped| over s in [-50, 500] km: "
                  f"{np.max(np.abs(a - b)):.3e}")
            print("  (0 means our emitted LuaMap is the same function as the shipped one)")
        except Exception as exc:
            print(f"  ! could not compare the Lua maps: {exc}")

## [6] Assemble + pre-flight

Into `decks/safs_alt_check/` — its own folder, never a shipped deck.

In [ ]:
from deckbuild.deck import DeckSpec, DeckStage, resolve_paths
arts = {"material": mat.material, "stress": stress, "friction": friction,
        "mesh": mesh_art}
if mat.plasticity is not None:
    arts["plasticity"] = mat.plasticity
spec = DeckSpec(prefix="safs_", plasticity=mat.plasticity is not None,
                attenuation=mat.attenuation, mu_s=0.6, end_time_s=150.0,
                rs_muw_lua=Path(fw.path).read_text(),
                nucleation_lua=nucleation_lua(snap, 2000.0, 75.0e6),
                notes="# Built by safs_check.ipynb -- a CHECK deck, not a production run.")
for k, e in resolve_paths(DECK, arts, spec).items():
    print(f"  {k:12} -> {e['filename']}")
dstage = DeckStage()
deck = dstage.assemble(cfg, DECK, arts, spec, overwrite=True)
dstage.preflight(cfg, deck, spec=spec).print()

## [7] Verdict

In [ ]:
print(f"deck : {deck}")
for f in sorted(deck.iterdir()):
    print(f"  {f.name:48} {f.stat().st_size/1e6:8.1f} MB")
if shipped:
    print(f"\nshipped deck for comparison: {shipped.name}")
    print("  file-by-file diff is NOT run here: the two decks use different meshes")
    print("  (ALT vs the deep40km variant) and different EndTime/output settings.")
    print("  The meaningful comparisons are the FIELD ones in [3] and [5] above.")
print("\nSee docs/EXERCISE_safs_reproduction.md for the E0-E6 ladder and what is")
print("still open.  The grid construction reproduces exactly; the CVM values do not.")